# Electrical stimulation fMRI (EPI and SORDINO): stitching and preprocessing

*Version 2.4 of the electrical stimulation notebooks (history in [CHANGELOG.md](CHANGELOG.md)). One notebook for EPI and SORDINO: the sequence is detected from the Bruker `method` file of the first functional scan (or chosen in the "Sequence" dropdown). It replaces `mol_fMRI_electric_stim_preprocessing.ipynb` (EPI) and `mol_fMRI_sordino_preprocessing.ipynb` (SORDINO), which are kept unchanged.*

1. Convert the raw Bruker functional scans (same session and subject, acquired back-to-back with identical acquisition parameters) to NIfTI and concatenate them in the given order.
   - **EPI:** at least 2 scans. The first volumes of every scan are dropped before stitching (default 10), because the signal of a new scan is not yet in steady state (a bright first volume that decays over a few volumes). Optionally, the stitched series is then corrected for susceptibility distortion with a reverse phase-encoded EPI (FSL topup, one field estimated from the first volume of the series).
   - **SORDINO** (zero echo time, 3D radial, no slices, no distortion correction): 1 scan or more. The converted NIfTIs carry a wrong TR (1 s, Bruker only stores the 0.75 ms spoke time), so the stitched series is written with the real volume TR (`PVM_ScanTime / PVM_NRepetitions`, e.g. 2.4 s for 3200 spokes). By default no volumes are dropped, because the 10 s of dummy scans before every sordino scan already bring the signal into steady state. The stimulation paradigm is not stored in the method file: the scanner only sends a trigger pulse to the stimulator at the start of the scan and at the start of every trigger block (`NVolumesPerBlock`). The trigger blocks of every scan are printed below and saved in `session.json` for the analysis.
2. Make the brain mask on the middle volume (hand-drawn or automatic, see *Brain Mask*).
3. Motion-correct the stitched series once against a single reference volume (the middle volume of the whole series), compute the mean motion-corrected image and process the structural scan (structural brain mask, hand-drawn or automatic).

Distortion correction (EPI only) is off by default; tick *Distortion Correction* in the pipeline steps at the top to use it, and enter the scan number of the reverse phase-encoded EPI. The phase-encoding axis and blip sign are detected from the Bruker metadata (the sign is only a best guess, so the detected values are shown and need to be confirmed before topup runs) or can be set manually. The corrected data is stored in its own `stitched_<scans>_<sequence>_distcorr` folder, so corrected and uncorrected runs never overwrite each other and the mask is made on the corrected images.

Optionally (step *Normalise Scan Levels*, off by default), the scans are scaled so that the signal level continues across the stitching points, which removes steps between scans caused by e.g. a change of the head position relative to the coil. One factor per scan is applied (`scan_normalization.csv`), so changes within a scan stay untouched. Real differences between scans, e.g. a lasting change after the stimulation, are removed as well, so keep the baseline of an analysis in the same scan as its signal window when this is used. The settings are part of the folder name (`..._distcorr_drop10_norm`), so runs with different settings never overwrite each other.

Enter the functional scan numbers in acquisition order. `scan_boundaries.csv` in the functional analysis folder records the volume range of each original scan.

Every step is skipped when its output already exists and is newer than its inputs, so the notebook can simply be run again. Tick *Force re-run* to recompute everything. The results go to the stitched folder (`func.nii.gz`, `mc_func.nii.gz`, masks) and to `<analysis folder name>` inside it (`mean_mc_func.nii.gz`, `session.json`). `session.json` is loaded by the analysis notebooks, which continue from here.

In [ ]:
import electric_stim_helpers, electric_stim_session, sordino_helpers, brain_masks
import mol_fMRI_distortion_correction_rp as distortion_correction
from electric_stim_helpers import *
from electric_stim_session import *
from sordino_helpers import *
from brain_masks import *
from fmri_paths import get_fmri_root

# Version of this notebook: saved in session.json. (STIM_NOTEBOOKS_VERSION in electric_stim_session.py stays at 2.3 for the
# older EPI/SORDINO preprocessing notebooks, which are kept unchanged.)
NOTEBOOK_VERSION = "2.4"
print(f"Electrical stimulation notebooks, version {NOTEBOOK_VERSION} (EPI + SORDINO preprocessing; history: CHANGELOG.md)")

In [ ]:
def int_widget(value, desc):
    return widgets.IntText(value=value, description=desc, layout=INPUT_LAYOUT, style={'description_width': DESC_WIDTH})


def print_selected_pipeline_steps():
    print("\nPIPELINE STEP SELECTION SUMMARY")
    print("-" * 40)
    for step in PIPELINE_STEPS:
        print(f"[{'✓' if checkboxes[step['key']].value else ' '}] {step['name']}")
    print("-" * 40)


def is_step_enabled(step_key):
    return checkboxes.get(step_key, None) and checkboxes[step_key].value


PIPELINE_STEPS = [
    {"name": "Bruker → NIfTI + Stitching", "key": "bruker_to_nifti"},
    {"name": "Distortion Correction", "key": "distortion_correction", "default": False},
    {"name": "Motion Correction", "key": "motion_correction"},
    {"name": "Masking", "key": "masking_file"},
    {"name": "Normalise Scan Levels", "key": "scan_normalization", "default": False}]

checkboxes = {step["key"]: widgets.Checkbox(value=step.get("default", True), description=step["name"], indent=False) for step in PIPELINE_STEPS}

display(widgets.Box(list(checkboxes.values()), layout=widgets.Layout(display="flex", flex_flow="row wrap", align_items="flex-start", gap="10px")))

In [ ]:
## Make sure what functions are selected to run in the pipeline
print_selected_pipeline_steps()

# Data root: environment variable FMRI_ROOT, fmri_root.txt or the folder containing this repository (see fmri_paths.py)
FMRI_ROOT = get_fmri_root()

## Choosing the raw data that needs to be analysed
root_location = f"{FMRI_ROOT}/RawData"
selected_folder = FileChooser(root_location, layout=widgets.Layout(width='1080px'))
selected_folder.show_only_dirs = True
display(selected_folder)

In [ ]:
in_path = Path(selected_folder.selected_path)
print(f"Our Raw Data is located in the path {in_path}")
subject_id = extract_subject_id(in_path)

# ---------- Common styles ----------
INPUT_LAYOUT = widgets.Layout(width="280px", flex="0 0 auto")
ROW_LAYOUT = widgets.Layout(width="100%", justify_content="space-between", align_items="center", padding="12px", border="3px solid #444", border_radius="10px", background_color="#1e1e1e")
BOX_LAYOUT = widgets.Layout(width="100%", flex_flow="row wrap", justify_content="space-between", align_items="center", padding="12px", border="3px solid #444", border_radius="10px", background_color="#1e1e1e")
DESC_WIDTH = "220px"

# ---------- Widgets (KEEP REFERENCES) ----------
func_scan_numbers_entered = widgets.Text(value="", placeholder="e.g. 31 or 10, 11, 12", description="Functional Scans (in order):", layout=INPUT_LAYOUT, style={'description_width': DESC_WIDTH})
struct_scan_number_entered = int_widget(10, "Structural Scan:")
# "auto" detects EPI or SORDINO from the Bruker method file of the first functional scan; pick one to override the detection
sequence_entered = widgets.Dropdown(options=["auto", "EPI", "SORDINO"], value="auto", description="Sequence:", layout=INPUT_LAYOUT, style={'description_width': DESC_WIDTH})
# empty = default of the sequence (EPI 10, SORDINO 0)
drop_vols_entered = widgets.Text(value="", placeholder="EPI 10, SORDINO 0", description="Volumes to drop per scan:", layout=INPUT_LAYOUT, style={'description_width': DESC_WIDTH})

row = widgets.HBox([func_scan_numbers_entered, struct_scan_number_entered, sequence_entered, drop_vols_entered], layout=BOX_LAYOUT)
mask_source_dir_entered = widgets.Text(value="", placeholder="optional", description="Reuse masks from folder:", layout=INPUT_LAYOUT, style={'description_width': DESC_WIDTH})
analysis_name_entered = widgets.Text(value="analysis", description="Analysis folder name:", layout=INPUT_LAYOUT, style={'description_width': DESC_WIDTH})
force_rerun_entered = widgets.Checkbox(value=False, description="Force re-run of finished steps", indent=False)
row_folders = widgets.HBox([mask_source_dir_entered, analysis_name_entered, force_rerun_entered], layout=ROW_LAYOUT)

# Source of the brain masks (only used if the Masking step is ticked). Existing masks are never replaced.
mask_style = {'description_width': '120px'}
struct_mask_source_entered = widgets.Dropdown(options=[("hand-drawn (fsleyes)", "hand"), ("automatic (3dSkullStrip + 3dAutomask)", "auto")], value="hand",
                                              description="Structural mask:", layout=widgets.Layout(width="360px"), style=mask_style)
func_mask_source_entered = widgets.Dropdown(options=[("hand-drawn (fsleyes)", "hand"), ("automatic (from the structural mask)", "auto")], value="hand",
                                            description="Functional mask:", layout=widgets.Layout(width="400px"), style=mask_style)
# SORDINO only: the 3D sordino volume contains the whole head, so the automatic masks cover a slab of the structural slices, centred on
# the widest brain slice and shifted towards anterior (default 20 slices, +3: the striatum-centred slab of the hand-drawn masks)
sordino_mask_slices_entered = widgets.IntText(value=20, description="SORDINO slab (slices):", layout=widgets.Layout(width="260px"), style={'description_width': '170px'})
sordino_mask_shift_entered = widgets.IntText(value=3, description="slab shift (slices, + = anterior):", layout=widgets.Layout(width="300px"), style={'description_width': '220px'})
mask_note = widgets.HTML("<b>Automatic masks are still in testing</b> (methods from mol_fMRI_v5.8, not yet tested on stimulation sessions). Structural: accurate in tests "
                         "(overlap with hand-drawn masks 0.94-0.97), usable after a look at the quality-check figure. "
                         "Functional: made from the structural mask (EPI: 0.89, down to 0.7; SORDINO: 0.87-0.93 inside the slab, 3 sessions); the quality check cannot "
                         "reliably tell whether it is good: <b>always check the figure</b>, and draw the mask by hand if it is not good. "
                         "The structural mask needs a standard anatomy of the same session (e.g. 256 x 256, 25.6 mm, 32 slices; on 700 x 700 scans 3dSkullStrip failed). "
                         "SORDINO slab: the automatic masks cover this many structural slices (0 = all), centred on the widest brain slice and shifted towards anterior. Tested on rat data only.")
row_masks = widgets.VBox([widgets.HBox([struct_mask_source_entered, func_mask_source_entered]),
                          widgets.HBox([sordino_mask_slices_entered, sordino_mask_shift_entered]), mask_note],
                         layout=widgets.Layout(width="100%", padding="12px", border="3px solid #444", border_radius="10px", background_color="#1e1e1e"))

# Only used if the Distortion Correction step is selected (EPI only)
rev_pe_scan_number_entered = widgets.Text(value="", placeholder="e.g. 12", description="Reverse-PE Scan:", layout=INPUT_LAYOUT, style={'description_width': DESC_WIDTH})
pe_axis_entered = widgets.Dropdown(options=["auto", "x", "y", "z"], value="auto", description="Phase-encoding axis:", layout=INPUT_LAYOUT, style={'description_width': DESC_WIDTH})
blip_sign_entered = widgets.Dropdown(options=["auto", "+1", "-1"], value="auto", description="Blip sign of functional scans:", layout=INPUT_LAYOUT, style={'description_width': DESC_WIDTH})
readout_time_entered = widgets.FloatText(value=0.05, description="Total readout time (s):", layout=INPUT_LAYOUT, style={'description_width': DESC_WIDTH})
row_distortion = widgets.HBox([rev_pe_scan_number_entered, pe_axis_entered, blip_sign_entered, readout_time_entered], layout=BOX_LAYOUT)
display(row, row_folders, row_masks, row_distortion)

In [ ]:
func_scan_numbers = [s for s in re.split(r"[,\s]+", func_scan_numbers_entered.value.strip()) if s]
if not func_scan_numbers:
    raise ValueError("Enter the functional scan numbers in acquisition order, e.g. '31' or '10, 11, 12'.")
func_scan_label = "-".join(func_scan_numbers)
struct_scan_number = str(struct_scan_number_entered.value)
mask_source_dir = mask_source_dir_entered.value.strip() or None
struct_mask_source = struct_mask_source_entered.value  # "hand" or "auto"
func_mask_source = func_mask_source_entered.value  # "hand" or "auto"

# ---- Sequence type: "Sequence" dropdown; "auto" = detect from the Bruker method file of the first scan, ask if that fails ----
detected = {num: detect_sequence_type(Path(in_path) / num) for num in func_scan_numbers}
detected_sequence = detected[func_scan_numbers[0]]
sequence_type = None if sequence_entered.value == "auto" else sequence_entered.value.lower()
if sequence_type is None:
    sequence_type = detected_sequence
    if sequence_type is None:
        answer = input("The sequence could not be detected from the method file. Are the functional scans SORDINO or EPI? [sordino/epi]: ").strip().lower()
        sequence_type = {"s": "sordino", "sordino": "sordino", "e": "epi", "epi": "epi"}.get(answer)
        if sequence_type is None:
            raise ValueError("Unknown answer: enter 'sordino' or 'epi', or choose the sequence in the 'Sequence' dropdown.")
elif detected_sequence is not None and sequence_type != detected_sequence:
    print_statement(f"Note: the Sequence dropdown says {sequence_type.upper()}, but the method file of scan {func_scan_numbers[0]} looks like {detected_sequence.upper()}.", bcolors.FAIL)
for num, seq in detected.items():
    if seq is not None and seq != sequence_type:
        print_statement(f"Scan {num} looks like {seq.upper()}, not {sequence_type.upper()}: all scans that are stitched must have the same sequence.", bcolors.FAIL)
is_sordino = sequence_type == "sordino"
print_statement(f"Functional sequence: {'SORDINO (zero echo time, 3D radial)' if is_sordino else 'EPI'}"
                f"{' (detected from the method file)' if sequence_entered.value == 'auto' and detected_sequence is not None else ''}", bcolors.NOTIFICATION)
if not is_sordino and len(func_scan_numbers) < 2:
    raise ValueError("EPI: enter at least 2 functional scan numbers to stitch, e.g. '10, 11, 12'.")

drop_text = drop_vols_entered.value.strip()
drop_vols = max(0, int(drop_text)) if drop_text else (0 if is_sordino else 10)
normalise_enabled = bool(is_step_enabled("scan_normalization"))

# SORDINO: automatic masks cover a slab of the structural slices (the 3D sordino volume contains the whole head); EPI: the functional
# slices already limit the mask
sordino_mask_slices = int(sordino_mask_slices_entered.value)
sordino_mask_shift = int(sordino_mask_shift_entered.value)
auto_mask_slab = {"n_slices": sordino_mask_slices or None, "shift_slices": sordino_mask_shift} if is_sordino else {}

distortion_enabled = bool(is_step_enabled("distortion_correction"))
rev_pe_scan_number = rev_pe_scan_number_entered.value.strip()
pe_axis = None if pe_axis_entered.value == "auto" else pe_axis_entered.value
blip_sign = None if blip_sign_entered.value == "auto" else int(blip_sign_entered.value)
readout_time = float(readout_time_entered.value)
if distortion_enabled:
    if is_sordino:
        raise ValueError("Distortion Correction is selected, but sordino (zero echo time, 3D radial) has no EPI distortion. Untick the step.")
    if not rev_pe_scan_number:
        raise ValueError("Distortion Correction is selected: enter the reverse phase-encoded scan number, or untick the step.")
    if rev_pe_scan_number in func_scan_numbers:
        raise ValueError(f"Reverse-PE scan {rev_pe_scan_number} is also one of the functional scans.")
    if not distortion_correction.is_bruker_raw_scan_dir(Path(in_path) / rev_pe_scan_number):
        raise FileNotFoundError(f"Reverse-PE scan {rev_pe_scan_number} not found in {in_path} (no 'method' file).")

print_statement(f"Subject under analysis is {subject_id} for Functional Scans {', '.join(func_scan_numbers)} (stitched in this order) and Structural Scan Number {struct_scan_number}", bcolors.NOTIFICATION)
print_statement(f"Dropping the first {drop_vols} volumes of every scan; scan level normalisation {'on' if normalise_enabled else 'off'}.", bcolors.NOTIFICATION)
if distortion_enabled:
    print_statement(f"Distortion correction with reverse-PE scan {rev_pe_scan_number} (PE axis: {pe_axis or 'auto'}, blip sign: {blip_sign or 'auto'}, readout time: {readout_time} s)", bcolors.NOTIFICATION)
elif not is_sordino:
    print_statement("Distortion correction is not selected.", bcolors.NOTIFICATION)
print_statement(f"Brain masks: structural {'automatic' if struct_mask_source == 'auto' else 'hand-drawn'}, functional {'automatic' if func_mask_source == 'auto' else 'hand-drawn'} "
                f"(existing masks are used as they are){f'; SORDINO slab {sordino_mask_slices} slices, shift {sordino_mask_shift:+d}' if is_sordino and func_mask_source == 'auto' else ''}.", bcolors.NOTIFICATION)

# # Parameters extraction for structural and functional files

params_struct = func_param_extract(Path(in_path) / struct_scan_number,export_env=True)
sequence_name_struct = params_struct["SequenceName"]

if is_sordino:
    params_all = {num: sordino_param_extract(Path(in_path) / num) for num in func_scan_numbers}
else:
    params_all = {num: func_param_extract(Path(in_path) / num, export_env=(i == 0)) for i, num in enumerate(func_scan_numbers)}
params_func = params_all[func_scan_numbers[0]]
sequence_name_func = params_func["SequenceName"]

# Stitching assumes identical acquisition parameters across scans; sordino scans with block triggers can differ by a dummy spoke per
# volume (0.75 ms)
tr_tolerance = 0.01 if is_sordino else 1e-3
for num, p in params_all.items():
    if is_sordino and not p["is_sordino"]:
        print_statement(f"Scan {num} is not a sordino scan (method {p['Method']}).", bcolors.FAIL)
    if p["SequenceName"] != sequence_name_func or abs((p["VolTR"] or 0) - (params_func["VolTR"] or 0)) > tr_tolerance:
        print_statement(f"Scan {num} differs from scan {func_scan_numbers[0]} in sequence name or TR.", bcolors.FAIL)

if is_sordino:
    print_header("Sordino scans: volume TR and trigger blocks", bcolors.HEADER)
    for num, p in params_all.items():
        trigger_text = (f"start-of-scan trigger: {'yes' if p['StartOfScanTrigger'] else 'no'}, {len(p['block_onsets'])} block triggers every {sorted(set(p['volumes_per_block']))} volumes"
                        if p["BlockTriggerMode"] else "no triggers")
        print_statement(f"Scan {num}: {p['NoOfRepetitions']} volumes, TR {p['VolTR']:.5f} s, {trigger_text}", bcolors.NOTIFICATION)


# Setting up path for storing analysed data overall, structural data, functional data and processed functional data
# Analysed data overall
parts = list(in_path.parts)
parts[parts.index("RawData")] = "AnalysedData"
analysed_base = Path(*parts).parent
analysed_path = analysed_base / subject_id

# Structural and Functional data
analysed_struct_dir = Path(analysed_path) / f"{struct_scan_number}{sequence_name_struct}"
# the processing options are part of the folder name so results with different options never mix
dir_suffix = ("_distcorr" if distortion_enabled else "") + (f"_drop{drop_vols}" if drop_vols else "") + ("_norm" if normalise_enabled else "")
analysed_func_dir = Path(analysed_path) / f"stitched_{func_scan_label}_{sequence_name_func}{dir_suffix}"

# processed functional data: a fixed folder, so a later run (or the analysis notebook) finds the same files
analysis_dir = analysed_func_dir / (analysis_name_entered.value.strip() or "analysis")
analysis_dir.mkdir(parents=True, exist_ok=True)
electric_stim_session.FORCE_RERUN = force_rerun_entered.value

# path for raw structural and functional data
path_raw_struct = os.path.join(in_path, struct_scan_number)
path_raw_func = ", ".join(os.path.join(in_path, num) for num in func_scan_numbers)

# Display of variables and paths in tabular form
var_vals = [
            ['Location of all Raw Data', 'root_location', root_location],
            ['Location of Raw Data to be analysed', 'in_path', in_path],
            ['Location of Raw Structural Data', 'path_raw_struct', path_raw_struct],
            ['Location of Raw Functional Data', 'path_raw_func', path_raw_func],
            ['Location of analysed structural data', 'analysed_struct_dir', analysed_struct_dir],
            ['Location of analysed functional data', 'analysed_func_dir', analysed_func_dir],
            ['Location of final processed functional data', 'analysis_dir', analysis_dir]
            ]

# pd.set_option('display.max_colwidth', 500)  # or 199
df = pd.DataFrame(var_vals, columns=['Purpose', 'Variable Name', 'Value'], index=['a', 'b', 'c', 'd', 'e', 'f', 'g'])
print(tabulate(df, headers='keys', tablefmt='psql'))

# Converting Functional Scans into NIFTI and Stitching them

In [ ]:
print_statement(f"Analysed Data Path: {analysed_func_dir}", bcolors.NOTIFICATION)
analysed_func_dir.mkdir(parents=True, exist_ok=True)
os.chdir(analysed_func_dir)

if is_step_enabled("bruker_to_nifti"):
    print_header("Converting Bruker to NIFTI and stitching: Functional Data", bcolors.HEADER)

    stitched_file = analysed_func_dir / "func.nii.gz"

    if stitched_file.exists():
        print_statement("Stitched NIfTI file already exists. Skipping conversion and stitching.", bcolors.OKGREEN)
    else:
        scan_files = []
        for num in func_scan_numbers:
            work_dir = analysed_func_dir / "converted_raw" / f"scan_{num}"
            out_name = f"{num}_func_raw.nii.gz"
            if (work_dir / out_name).exists():
                print_statement(f"Scan {num} already converted. Skipping conversion.", bcolors.OKGREEN)
            else:
                bruker_scan_to_nifti(in_path, num, work_dir, out_name)
            if drop_vols:
                trimmed_name = f"{num}_func_raw_drop{drop_vols}.nii.gz"
                if not (work_dir / trimmed_name).exists():
                    drop_first_volumes(work_dir / out_name, drop_vols, work_dir / trimmed_name)
                out_name = trimmed_name
            scan_files.append(str((work_dir / out_name).resolve()))

        per_scan_volumes = [n_volumes(f) for f in scan_files]
        # With distortion correction the uncorrected stitched series is an intermediate; func.nii.gz is the corrected one
        stitched_raw_file = analysed_func_dir / ("func_uncorrected.nii.gz" if distortion_enabled else "func.nii.gz")
        if stitched_raw_file.exists() and not force_rerun_entered.value:
            print_statement(f"{stitched_raw_file.name} already exists. Skipping stitching.", bcolors.OKGREEN)
        elif is_sordino:
            # the converted sordino NIfTIs carry TR = 1 s; the stitched series gets the real volume TR: volume-weighted mean
            # (scans with block triggers have a dummy spoke per volume, +0.75 ms)
            stitched_tr = sum(params_all[num]["VolTR"] * n for num, n in zip(func_scan_numbers, per_scan_volumes)) / sum(per_scan_volumes)
            merge_with_tr(scan_files, stitched_raw_file.name, stitched_tr)
        else:
            subprocess.run(["fslmerge", "-t", stitched_raw_file.name] + scan_files, check=True)
            print_statement(f"[OK] Raw scans concatenated -> {stitched_raw_file.name}", bcolors.OKGREEN)
        write_boundaries_csv(scan_files, per_scan_volumes, "scan_boundaries.csv")

        if distortion_enabled:
            # One topup field for the whole stitched series, so all scans get the same correction
            print_header("Distortion correction of the stitched series (reverse phase-encoding, FSL topup)", bcolors.HEADER)
            corrected_file, = distortion_correction.run_with_confirmation(
                [stitched_raw_file], Path(in_path) / rev_pe_scan_number, analysed_func_dir / "distortion_correction",
                pe_axis=pe_axis, main_sign=blip_sign, readout_time=readout_time,
                skip_existing=not force_rerun_entered.value, detect_from=Path(in_path) / func_scan_numbers[0])
            shutil.copyfile(corrected_file, stitched_file)
            print_statement(f"[OK] Distortion corrected stitched series -> {stitched_file.name}", bcolors.OKGREEN)

else:
    print_statement("⏭️ Data Conversion Step not executed, May cause trouble in running further analysis", bcolors.NOTIFICATION)

input_name_for_next_step = "func"

# Brain Mask

The functional brain mask `mask_mean_mc_func.nii.gz` is taken, in this order, from: an existing mask in the functional folder; the folder in "Reuse masks from folder" (only valid if the scans have the same geometry); an automatic mask (if "Functional mask" is automatic); or drawn by hand in fsleyes on the middle volume.

The automatic functional mask (still in testing, methods from `mol_fMRI_v5.8`, functions in `brain_masks.py`) is made from the structural brain mask, which is made or drawn first ("Structural mask" dropdown; an existing `struct_mask.nii.gz` is used as it is). Both scans of a session share the scanner coordinates, so the structural mask is moved to the functional grid through the image headers only (no registration); it is wrong if the animal moved between the two scans. EPI: moved with nearest neighbour and trimmed with an intensity threshold. SORDINO: moved as a partial-volume fraction (a voxel is brain if at least 30% of it is covered), no trimming (sordino has almost no intensity edge at the brain border), and limited to the SORDINO slab of structural slices. The image for the mask is the mean of the 30 volumes around the middle of the stitched series (`auto_mask_reference.nii.gz`, before motion correction). An automatic mask is checked in `mask_qc_functional.png` (red = mask, cyan dashed = structural mask before the trimming): **look at the figure**; if the outline does not follow the brain, delete `mask_mean_mc_func.nii.gz` and draw it by hand. A hand-drawn mask saved over an automatic one counts as hand-drawn (`<mask>.source.json`).

In [ ]:
os.environ["FSLOUTPUTTYPE"] = "NIFTI_GZ"

# real volume TR: SORDINO from the header of the stitched series (written at stitching), EPI from the method file (float, so e.g. 1.5 s
# is not truncated)
tr = (float(subprocess.run(["fslval", "func.nii.gz", "pixdim4"], capture_output=True, text=True, check=True).stdout) if is_sordino
      else float(params_func["VolTR"]))
n_vols = n_volumes("func.nii.gz")

# Middle volume of the whole stitched series: reference for motion correction and the image the mask is drawn on
run_step(["middle_vol.nii.gz"], ["func.nii.gz"], extract_middle_volume, "func.nii.gz", int(n_vols / 2), "middle_vol.nii.gz", 1)

mask_file = os.path.join(analysed_func_dir, "mask_mean_mc_func.nii.gz")
auto_reference = os.path.join(analysed_func_dir, "auto_mask_reference.nii.gz")
auto_prior = os.path.join(analysed_func_dir, "auto_mask_structural_prior.nii.gz")

if is_step_enabled("masking_file"):
    # Only valid if the stitched scans share identical geometry with the mask source scan
    mask_src = Path(mask_source_dir) / "mask_mean_mc_func.nii.gz" if mask_source_dir else None
    if not os.path.exists(mask_file) and mask_src is not None and mask_src.exists():
        shutil.copyfile(mask_src, mask_file)
        print_statement(f"[OK] Reused mask from {mask_src}", bcolors.OKGREEN)

    # Automatic functional mask (still in testing): made from the structural mask; an existing mask is never replaced
    if func_mask_source == "auto" and not os.path.exists(mask_file):
        print_header("Automatic functional brain mask (still in testing)", bcolors.HEADER)
        struct_mask_for_func = prepare_structural_mask(in_path, struct_scan_number, analysed_struct_dir, struct_mask_source,
                                                       functional_reference=os.path.join(analysed_func_dir, "middle_vol.nii.gz"), **auto_mask_slab)
        mean_of_middle_volumes("func.nii.gz", auto_reference)
        # SORDINO: no intensity trimming (no intensity edge at the brain border), partial-volume transfer, slab of structural slices
        sordino_options = dict(trim_rel=None, partial_volume=True, **auto_mask_slab) if is_sordino else {}
        automatic_functional_mask(auto_reference, str(struct_mask_for_func), mask_file, prior_out=auto_prior, **sordino_options)

    if os.path.exists(mask_file):
        print_statement(f"Mask Image ({mask_file}) exists ({mask_source(mask_file)}).", bcolors.OKGREEN)
    else:
        print_statement("Mask Image does not exist. Please create the mask and save it as mask_mean_mc_func.nii.gz", bcolors.FAIL)
        subprocess.run(["fsleyes", "middle_vol.nii.gz"])
        require_files([mask_file], "Save the brain mask as mask_mean_mc_func.nii.gz in the functional analysis folder.")

    # Quality check of an automatic functional mask (figure + numbers); a hand-drawn mask is not checked
    if mask_source(mask_file) != "hand-drawn":
        print_header("Quality check of the automatic functional brain mask", bcolors.HEADER)
        qc_image = auto_reference if os.path.exists(auto_reference) else "middle_vol.nii.gz"
        mask_quality_check(qc_image, mask_file, "functional", os.path.join(analysed_func_dir, "mask_qc_functional.png"),
                           prior_file=auto_prior if os.path.exists(auto_prior) else None, title="Automatic functional brain mask",
                           edge_warning=not is_sordino)
        print_statement("The automatic functional mask is less accurate than the structural one and the checks above only catch gross failures: "
                        "LOOK AT THE FIGURE. If the outline does not follow the brain, delete mask_mean_mc_func.nii.gz and draw the mask by hand "
                        "(and re-run the steps after it).", bcolors.FAIL)
else:
    print_statement("⏭️ Masks not generated, will not get cleaned data in further pipeline", bcolors.NOTIFICATION)

# Scan Level Normalisation (optional)

Scales every scan so that its signal level continues the neighbouring scan (see the description at the top), starting from the scan that contains the middle volume, which is the motion correction reference. Needs the brain mask. The normalised series `func_norm.nii.gz` is the input of the motion correction; the factors are in `scan_normalization.csv`.

In [ ]:
if normalise_enabled:
    print_header("Normalising the signal level of the scans", bcolors.HEADER)
    require_files([mask_file], "Scan level normalisation needs the brain mask: tick Masking and save the mask first.")
    run_step(["func_norm.nii.gz", "scan_normalization.csv"], ["func.nii.gz", mask_file], normalise_scan_levels,
             "func.nii.gz", mask_file, read_scan_ranges(analysed_func_dir), int(n_vols / 2), "func_norm.nii.gz", "scan_normalization.csv")
else:
    print_statement("⏭️ Scan level normalisation not selected", bcolors.NOTIFICATION)

# Applying Motion Correction in Raw Functional Data and Plotting Motion Parameters

In [ ]:
motion_input = "func_norm.nii.gz" if normalise_enabled else "func.nii.gz"

if is_step_enabled("motion_correction"):
    print_header("Applying Motion Correction on raw functional data and plotting motion parameters", bcolors.HEADER)

    if os.path.exists("mc_func.nii.gz"):
        print_statement("Motion Corrected functional data exists. Skipping motion correction.", bcolors.OKGREEN)
    else:
        motion_correction("middle_vol.nii.gz", motion_input, output_prefix="mc_func")

    # Display of the motion corrected image and motion parameters
    view_images("mc_func.nii.gz", cmap="gray")
    plot_motion_parameters("motion.1D")
else:
    print_statement("⏭️ Motion Correction not executed", bcolors.NOTIFICATION)

# Mean Motion-Corrected Image

In [ ]:
print_statement(f"Analysis folder: {analysis_dir}", bcolors.NOTIFICATION)
analysis_dir.mkdir(parents=True, exist_ok=True)
os.chdir(analysis_dir)

mc_func_file = analysed_func_dir / "mc_func.nii.gz"
run_step(["mean_mc_func.nii.gz"], [mc_func_file], mean_image, str(mc_func_file), "mean_mc_func.nii.gz")

# Converting Structural Data into NIFTI and Processing it

In [ ]:
print_statement(f"Analysed Data Path: {analysed_struct_dir}", bcolors.NOTIFICATION)
analysed_struct_dir.mkdir(parents=True, exist_ok=True)
os.chdir(analysed_struct_dir)

if is_step_enabled("bruker_to_nifti"):
    print_header("Converting Bruker to NIFTI: Structural Data", bcolors.HEADER)

    nifti_file = analysed_struct_dir / "struct.nii.gz"
    if nifti_file.exists():
        print_statement("NIfTI file already exists. Skipping conversion.", bcolors.OKGREEN)
    else:
        bruker_to_nifti(in_path, struct_scan_number, "struct.nii.gz")

else:
    print_statement("⏭️ Data Conversion Step not executed, May cause trouble in running further analysis", bcolors.NOTIFICATION)


# Cleaning the structural image with its brain mask (hand-drawn in fsleyes or automatic, "Structural mask" dropdown; an existing
# struct_mask.nii.gz is used as it is)

if is_step_enabled("masking_file"):
    if os.path.exists("cleaned_struct.nii.gz"):
        print_statement("Structural Image for Coregistration exists.", bcolors.OKGREEN)
    else:
        struct_mask_path = prepare_structural_mask(in_path, struct_scan_number, analysed_struct_dir, struct_mask_source,
                                                   functional_reference=os.path.join(analysed_func_dir, "middle_vol.nii.gz"), **auto_mask_slab)
        masking_file("struct.nii.gz", str(struct_mask_path), "cleaned_struct.nii.gz")

    structural_file_for_coregistration = os.path.join(analysed_struct_dir, "cleaned_struct.nii.gz")

    # Quality check of an automatic structural mask (figure + numbers); a hand-drawn mask is not checked
    struct_mask_path = structural_mask_file(analysed_struct_dir)
    if struct_mask_path.exists() and mask_source(struct_mask_path) != "hand-drawn":
        print_header("Quality check of the automatic structural brain mask", bcolors.HEADER)
        mask_quality_check("struct.nii.gz", str(struct_mask_path), "structural", os.path.join(analysed_struct_dir, "mask_qc_structural.png"),
                           title="Automatic structural brain mask")
        print_statement("The automatic structural mask is accurate in tests (overlap with hand-drawn masks 0.94-0.97) and can be used after a look at the figure. "
                        "If it is not good, delete struct_mask.nii.gz and cleaned_struct.nii.gz and draw the mask by hand.", bcolors.OKGREEN)
    view_images("cleaned_struct.nii.gz", cmap="gray")

else:
    print_statement("⏭️ Masks not generated, will not get cleaned data in further pipeline", bcolors.NOTIFICATION)
    structural_file_for_coregistration = os.path.join(analysed_struct_dir, "struct.nii.gz")
    view_images("struct.nii.gz", cmap="gray")

In [ ]:
# The stimulation analysis only needs the motion corrected data, the mask and the structural image
struct_mask_path = structural_mask_file(analysed_struct_dir)
mask_sources = {"functional": mask_source(mask_file) if os.path.exists(mask_file) else None,
                "structural": mask_source(struct_mask_path) if struct_mask_path.exists() else None}
sequence_keys = (dict(scan_info={num: {k: p[k] for k in ("NoOfRepetitions", "VolTR", "BlockTriggerMode", "StartOfScanTrigger", "TriggerPulseDuration_ms", "volumes_per_block", "block_onsets")}
                                 for num, p in params_all.items()})
                 if is_sordino else dict(distortion_correction=distortion_enabled, rev_pe_scan_number=rev_pe_scan_number or None))
session_file = save_session(
    analysis_dir,
    in_path=in_path, subject_id=subject_id, func_scan_numbers=func_scan_numbers, struct_scan_number=struct_scan_number,
    analysed_func_dir=analysed_func_dir, analysed_struct_dir=analysed_struct_dir, analysis_dir=analysis_dir,
    mask_file=mask_file, structural_file_for_coregistration=structural_file_for_coregistration, tr=tr,
    drop_vols=drop_vols, scan_normalization=str(analysed_func_dir / "scan_normalization.csv") if normalise_enabled else None,
    sequence_type=sequence_type, mask_sources=mask_sources, stim_notebooks_version=NOTEBOOK_VERSION, **sequence_keys)
print_statement(f"[OK] Session saved: {session_file}", bcolors.OKGREEN)